In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
# Evita notación científica al mostrar DataFrames
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")



In [2]:
# cargar tablon_analitico.pickle en df
ruta_pickle = Path('..') / 'datos' / 'intermedios' / 'tablon_analitico.pickle'
df = pd.read_pickle(ruta_pickle.resolve())

In [3]:
df.head()

,usuario,sesion,evento,categoria,producto,precio,año,mes,dia,hora,...,trimestre,fecha,festivo,dia_singles_11nov,dia_unidad_nacional_4nov,black_friday_2019,cyber_monday_2019,periodo_fin_ano_nuevo_ano,navidad_ortodoxa_7ene,defensor_patria_23feb
fecha_hora,,,,,,,,,,,,,,,,,,,,,
2019-10-01 00:01:46+00:00,462033176,a18e0999-61a1-4218-8f8f-61ec1d375361,view,1487580005092295511,5843665,9.44,2019,10,1,0,...,4,2019-10-01,0,0,0,0,0,0,0,0
2019-10-01 00:01:55+00:00,514753614,e2fecb2d-22d0-df2c-c661-15da44b3ccf1,cart,1487580013069861041,5868461,3.57,2019,10,1,0,...,4,2019-10-01,0,0,0,0,0,0,0,0
2019-10-01 00:02:50+00:00,527418424,86e77869-afbc-4dff-9aa2-6b7dd8c90770,view,1487580006300255120,5877456,122.22,2019,10,1,0,...,4,2019-10-01,0,0,0,0,0,0,0,0
2019-10-01 00:03:41+00:00,555448072,b5f72ceb-0730-44de-a932-d16db62390df,view,1487580013749338323,5649270,6.19,2019,10,1,0,...,4,2019-10-01,0,0,0,0,0,0,0,0
2019-10-01 00:03:44+00:00,552006247,2d8f304b-de45-4e59-8f40-50c603843fe5,view,1487580005411062629,18082,16.03,2019,10,1,0,...,4,2019-10-01,0,0,0,0,0,0,0,0


In [4]:
# Nos quedamos con los eventos de purchase
df = df[df['evento'] == 'purchase']
# Nos quedamos con las columnas que nos interesan
df = df[['usuario', 'producto']]
df.head()


,usuario,producto
fecha_hora,,
2019-10-01 00:26:49+00:00,536128518,5887011
2019-10-01 00:26:49+00:00,536128518,34767
2019-10-01 00:26:49+00:00,536128518,34768
2019-10-01 00:26:49+00:00,536128518,5584
2019-10-01 00:26:49+00:00,536128518,5588612


In [8]:
# Quitamos el índice fecha_hora y cramos una matriz temporal de usuarios y productos
temp = df.reset_index(drop=True)
temp

,usuario,producto
0,536128518,5887011
1,536128518,34767
2,536128518,34768
3,536128518,5584
4,536128518,5588612
...,...,...
127559,622065819,5659905
127560,622065819,5659911
127561,622065819,5697535
127562,622065819,5699414


In [9]:
# Vamos a ver el número de productos que tenemos
temp.producto.nunique()

23477

In [10]:
# Vamos a ver el número de productos que tenemos
temp.usuario.nunique()

11040

In [14]:
# Para evitar saturar el modelo con demasiados productos, nos quedamos con los 100 productos mas comprados
# sobre temp creamos una lista con los 100 productos mas comprados
top_productos = temp['producto'].value_counts().head(100).index.tolist()
top_productos

[5809910,
 5854897,
 5700037,
 5304,
 5751422,
 5802432,
 5809912,
 5815662,
 5751383,
 5792800,
 5849033,
 5686925,
 5700046,
 5528035,
 5833330,
 5809911,
 5816170,
 5687151,
 5729864,
 5843836,
 5013,
 5759492,
 5622677,
 5833334,
 5800788,
 5793704,
 5561044,
 5700035,
 5549834,
 5862943,
 5587740,
 5670733,
 5783987,
 5789668,
 5615144,
 5761411,
 5833326,
 4958,
 4497,
 5773361,
 5817702,
 5776130,
 5773605,
 5857360,
 4938,
 5809871,
 5848909,
 5700039,
 5759279,
 5814046,
 5649219,
 5759491,
 5816166,
 5528034,
 5809297,
 5693501,
 5723529,
 5773606,
 5842141,
 5677043,
 5688124,
 5754853,
 5751742,
 5816172,
 5889300,
 5528051,
 5857007,
 5784897,
 5793703,
 5763238,
 5790563,
 5815730,
 5833325,
 5622687,
 5848387,
 5835924,
 5810157,
 5526,
 5836522,
 5565820,
 4768,
 5585656,
 5824810,
 5550302,
 5585658,
 4600,
 5816169,
 5854812,
 5810480,
 5742957,
 5844894,
 5833318,
 5855332,
 5724230,
 5809303,
 5749149,
 5835859,
 5749720,
 5862564,
 5833335]

In [16]:
# filtra temp para quedarte solo con los productos en top_productos
temp_top = temp.loc[temp['producto'].isin(top_productos)].reset_index(drop=True)
temp_top

,usuario,producto
0,536128518,5815662
1,465338762,5817702
2,555497689,5649219
3,539216862,5854897
4,507864835,5622687
...,...,...
14211,582074095,5759491
14212,435564251,5802432
14213,617695004,5759491
14214,439781446,5742957


In [17]:
# Pivota producto a columnas y usuario a filas y las celdas tendrán el conteo de veces que el usuario compró el producto
matriz_usuario_item = temp_top.pivot_table(index='usuario', columns='producto', aggfunc='size', fill_value=0)
matriz_usuario_item

producto,4497,4600,4768,4938,4958,5013,5304,5526,5528034,5528035,...,5848909,5849033,5854812,5854897,5855332,5857007,5857360,5862564,5862943,5889300
usuario,,,,,,,,,,,,,,,,,,,,,
25392526,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
50748978,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
74332980,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
80577370,0,0,0,0,0,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
88211255,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
621646584,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
621788730,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
621925941,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [18]:
# Calcula la matriz de similaridad ítem-ítem usando distancia euclídea
from sklearn.metrics.pairwise import euclidean_distances

# Transpone la matriz para que los productos sean las filas
matriz_item_usuario = matriz_usuario_item.T

# Calcula la matriz de distancias euclídeas entre productos
distancia_euclid = euclidean_distances(matriz_item_usuario)

# Convierte a DataFrame para mejor visualización
similitud_item_item = pd.DataFrame(distancia_euclid, index=matriz_item_usuario.index, columns=matriz_item_usuario.index)

# guarda la matriz de similaridad ítem-ítem como pickle en datos/procesados/
similitud_item_item.to_pickle("../datos/procesados/similitud_item_item.pickle")

In [19]:
# carga similitud_item_item desde pickle desde datos/procesados/
similitud_item_item = pd.read_pickle("../datos/procesados/similitud_item_item.pickle")

In [20]:
similitud_item_item

producto,4497,4600,4768,4938,4958,5013,5304,5526,5528034,5528035,...,5848909,5849033,5854812,5854897,5855332,5857007,5857360,5862564,5862943,5889300
producto,,,,,,,,,,,,,,,,,,,,,
4497,0.00,14.42,14.49,15.62,15.91,17.58,23.39,15.23,16.16,20.30,...,15.72,24.45,14.97,26.10,15.30,15.07,15.78,15.17,16.40,15.33
4600,14.42,0.00,10.68,14.49,14.59,16.46,22.69,13.93,14.87,19.70,...,14.39,23.87,13.49,25.75,13.78,13.60,14.46,13.86,14.73,13.89
4768,14.49,10.68,0.00,14.56,14.73,16.28,22.74,14.00,14.93,19.75,...,14.18,24.00,13.27,25.63,13.86,13.75,14.39,13.86,14.80,14.11
4938,15.62,14.49,14.56,0.00,15.52,16.03,22.96,14.76,15.65,20.25,...,15.13,24.17,14.35,25.87,14.76,14.59,15.46,14.97,15.72,15.13
4958,15.91,14.59,14.73,15.52,0.00,17.49,22.27,15.33,15.81,20.32,...,15.49,24.35,14.66,26.15,14.87,14.97,15.94,15.26,16.12,15.03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5857007,15.07,13.60,13.75,14.59,14.97,16.79,22.80,14.59,15.10,19.62,...,14.49,23.98,13.89,25.61,14.32,0.00,14.83,14.46,15.17,14.42
5857360,15.78,14.46,14.39,15.46,15.94,16.06,23.24,15.39,16.12,20.32,...,15.43,23.85,14.73,25.46,15.20,14.83,0.00,13.75,16.19,15.43
5862564,15.17,13.86,13.86,14.97,15.26,15.72,22.91,14.70,15.39,19.90,...,15.00,23.32,14.00,25.12,14.35,14.46,13.75,0.00,15.39,14.66


In [21]:
def recomendar_productos(producto_id, matriz_similitud=similitud_item_item, n=5):
    """
    Devuelve los n productos más similares al producto dado usando la matriz de distancias euclídeas.
    Args:
        producto_id: ID del producto a comparar
        matriz_similitud: DataFrame de distancias euclídeas entre productos
        n: número de productos similares a devolver
    Returns:
        Lista de IDs de productos más similares
    """
    if producto_id not in matriz_similitud.index:
        raise ValueError("El producto no está en la matriz de similaridad")
    # Ordena por distancia creciente (más parecido)
    similares = matriz_similitud.loc[producto_id].sort_values()
    # El primero será el propio producto, así que lo excluimos
    similares = similares.iloc[1:n+1]
    return similares.index.tolist()

# Ejemplo de uso:
# recomendar_productos(4497, similitud_item_item, n=5)

In [22]:
temp_top.head()

,usuario,producto
0,536128518,5815662
1,465338762,5817702
2,555497689,5649219
3,539216862,5854897
4,507864835,5622687


In [23]:
recomendar_productos(5854897)

[5862564, 5817702, 5835924, 5810157, 5688124]

In [24]:
# Ahora vamos a crear una función que nos devuelva los productos más similares a una lista de productos


def recomendar_productos_varios(productos_ids, matriz_similitud=similitud_item_item, n=5):
    """
    Devuelve los n productos más similares considerando varios productos de entrada.
    Args:
        productos_ids: lista de IDs de productos a comparar
        matriz_similitud: DataFrame de distancias euclídeas entre productos
        n: número de productos similares a devolver
    Returns:
        Lista de IDs de productos más similares
    """
    # Filtra solo los productos que están en la matriz
    productos_validos = [p for p in productos_ids if p in matriz_similitud.index]
    if not productos_validos:
        raise ValueError("Ninguno de los productos está en la matriz de similaridad")
    # Calcula la distancia media de cada producto respecto a los productos de entrada
    distancias_medias = matriz_similitud.loc[productos_validos].mean(axis=0)
    # Excluye los productos de entrada
    distancias_medias = distancias_medias.drop(productos_validos)
    # Ordena por distancia creciente y devuelve los n más similares
    similares = distancias_medias.sort_values().iloc[:n]
    return similares.index.tolist()

# Ejemplo de uso:
# recomendar_productos_varios([4497, 4600], similitud_item_item, n=5)